In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

In [7]:
df = pd.read_csv('data/FC_plus_RES_withPredictions.csv')

In [8]:
# One-hot encoding
def one_hot_encode(seq):
    """Convert sequence to (4, seq_len) array"""
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    encoded = np.zeros((4, len(seq)), dtype=np.float32)
    for i, nuc in enumerate(seq):
        if nuc in mapping:
            encoded[mapping[nuc], i] = 1.0
    return encoded

In [9]:
sequences = df['30mer'].values
y = df['score_drug_gene_rank'].values

In [10]:
X_encoded = np.array([one_hot_encode(seq) for seq in sequences])
y = y.astype(np.float32)

In [12]:
print(f"X shape: {X_encoded.shape}")

X shape: (5310, 4, 30)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=51)

In [15]:
# Convert to tensors
X_train_t = torch.tensor(X_train)
X_test_t = torch.tensor(X_test)
y_train_t = torch.tensor(y_train).unsqueeze(1)
y_test_t = torch.tensor(y_test).unsqueeze(1)

In [16]:
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [21]:
# CNN Model Definition
class GuideCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Conv layers : input is (batch, 4, 30)
        self.conv1 = nn.Conv1d(in_channels=4, out_channels=64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(in_channels=128, out_channels=128, kernel_size=5, padding=2)

        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)

        # After conv1 + pool: 30 -> 15
        # After conv1 + pool: 15-> 7
        # After conv1 + pool: 7 -> 3
        # Flatten: 128 * 3 = 384

        self.fc1 = nn.Linear(128*3, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # x: (batch, 4, 30)
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))

        x = x.view(x.size(0), -1) # Flatten
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [22]:
# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = GuideCNN().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: cpu


In [23]:
# Training loop
n_epochs = 50
for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        # Evaluate every 10 epochs
        if (epoch+1)%10==0:
            model.eval()
            with torch.no_grad():
                preds = model(X_test_t.to(device)).cpu().numpy().flatten()
                spearman, _ = spearmanr(preds, y_test)
            print(f"Epoch {epoch+1}/{n_epochs} | Loss: {epoch_loss/len(train_loader):.4f} | Spearman: {spearman:.3f}")

Epoch 10/50 | Loss: 0.0005 | Spearman: 0.575
Epoch 10/50 | Loss: 0.0009 | Spearman: 0.577
Epoch 10/50 | Loss: 0.0014 | Spearman: 0.580
Epoch 10/50 | Loss: 0.0019 | Spearman: 0.582
Epoch 10/50 | Loss: 0.0024 | Spearman: 0.585
Epoch 10/50 | Loss: 0.0027 | Spearman: 0.585
Epoch 10/50 | Loss: 0.0032 | Spearman: 0.586
Epoch 10/50 | Loss: 0.0037 | Spearman: 0.586
Epoch 10/50 | Loss: 0.0042 | Spearman: 0.586
Epoch 10/50 | Loss: 0.0047 | Spearman: 0.588
Epoch 10/50 | Loss: 0.0051 | Spearman: 0.589
Epoch 10/50 | Loss: 0.0055 | Spearman: 0.591
Epoch 10/50 | Loss: 0.0061 | Spearman: 0.591
Epoch 10/50 | Loss: 0.0065 | Spearman: 0.590
Epoch 10/50 | Loss: 0.0069 | Spearman: 0.587
Epoch 10/50 | Loss: 0.0073 | Spearman: 0.585
Epoch 10/50 | Loss: 0.0078 | Spearman: 0.584
Epoch 10/50 | Loss: 0.0083 | Spearman: 0.583
Epoch 10/50 | Loss: 0.0087 | Spearman: 0.583
Epoch 10/50 | Loss: 0.0092 | Spearman: 0.584
Epoch 10/50 | Loss: 0.0095 | Spearman: 0.584
Epoch 10/50 | Loss: 0.0100 | Spearman: 0.583
Epoch 10/5

In [24]:
# Final evaluation
model.eval()
with torch.no_grad():
    y_pred_cnn = model(X_test_t.to(device)).cpu().numpy().flatten()

In [26]:
cnn_spearman, _ = spearmanr(y_pred_cnn, y_test)
print(f"\n{'='*40}")
print(f"Baseline (features + GB): 0.602")
print(f"CNN:                      {cnn_spearman:.3f}")
print(f"Azimuth:                  0.715")


Baseline (features + GB): 0.602
CNN:                      0.638
Azimuth:                  0.715
